# RGB300支撑评测

- 对应论文章节：第3章 RAG算法实验分析
- 源脚本：`experiments/02_消融实验/scripts/运行_RGB300支撑评测.py`
- notebook 作用：直接查看代码与已保存结果，命令行运行仍以 `.py` 为准

这本 notebook 对应 RGB300 稳定性支撑实验，以及和 CRUD 对照的图表输出。

## 命令行复现

```bash
cd /root/Velo
/root/Velo/.venv/bin/python experiments/02_消融实验/scripts/运行_RGB300支撑评测.py
```

## 源码镜像

下面这一格保留 `.py` 的完整源码，主要用于现场查阅。

In [ ]:
"""生成 RGB 300 稳定性支撑结果，并保留一份 CRUD 辅助对照表。"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path
from typing import Any, Sequence

import matplotlib.pyplot as plt
import nbformat as nbf
import pandas as pd
from matplotlib import font_manager

ROOT = Path(__file__).resolve().parents[1]
EXPERIMENTS_ROOT = ROOT.parent
IMPL_ROOT = EXPERIMENTS_ROOT / "04_算法实现"
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))

from retrieval_pipeline.common import DEFAULT_EMBEDDING_MODEL, DEFAULT_LLM_MODEL, ensure_dir
from retrieval_pipeline.datasets import load_crud_cases, load_rgb_cases
from retrieval_pipeline.metrics import DatasetEvaluation, evaluate_crud_results, evaluate_rgb_results, evenly_spaced_case_ids
from retrieval_pipeline.pipeline import PipelineVariant, RagExperimentPipeline

OUTPUT_ROOT = ROOT / "results" / "03_RGB300支撑结果"
NOTEBOOK_ROOT = ROOT / "results" / "_辅助notebook"
FONT_PATH = EXPERIMENTS_ROOT / ".assets" / "fonts" / "SourceHanSansSC-Regular.otf"

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "SimHei", "WenQuanYi Zen Hei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "path"


VARIANTS = (
    PipelineVariant(
        key="baseline_rrf_rerank_direct",
        label="基线（Dense + BM25 -> RRF -> Rerank -> Direct Answer）",
        use_rerank=True,
        answer_prompt_style="simple",
        multi_snippet_count=1,
    ),
    PipelineVariant(
        key="enhanced_dual_snippet_direct",
        label="增强检索（双片段证据保留）",
        use_rerank=True,
        answer_prompt_style="simple",
        multi_snippet_count=2,
    ),
    PipelineVariant(
        key="ours_task_router_aspect_cover_v2",
        label="主线方案（分项重排 + 覆盖取证 + 按题作答）",
        use_rerank=True,
        rerank_mode="aspect_aware_conservative",
        answer_prompt_style="task_router",
        multi_snippet_count=2,
        final_source_count=4,
        complex_source_count=6,
        selection_mode="aspect_cover_v2",
    ),
)


def load_cjk_font() -> font_manager.FontProperties | None:
    if not FONT_PATH.exists():
        return None
    font_manager.fontManager.addfont(str(FONT_PATH))
    return font_manager.FontProperties(fname=str(FONT_PATH))


def to_markdown_table(df: pd.DataFrame, columns: Sequence[str]) -> str:
    headers = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body: list[str] = []
    for _, row in df[list(columns)].iterrows():
        values: list[str] = []
        for value in row.tolist():
            if isinstance(value, float):
                values.append(f"{value:.4f}")
            else:
                values.append(str(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([headers, divider] + body)


def add_deltas(df: pd.DataFrame, metrics: Sequence[str]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    previous_row: dict[str, Any] | None = None
    for _, row in df.iterrows():
        record = row.to_dict()
        for metric in metrics:
            delta_key = f"delta_{metric}_vs_prev"
            if previous_row is None:
                record[delta_key] = "baseline"
            else:
                record[delta_key] = f"{float(record[metric]) - float(previous_row[metric]):+.4f}"
        rows.append(record)
        previous_row = record
    return pd.DataFrame(rows)


def plot_dataset_metrics(df: pd.DataFrame, output_path: Path, *, dataset_label: str, metrics: Sequence[tuple[str, str]]) -> None:
    cjk_font = load_cjk_font()
    figure, axes = plt.subplots(1, len(metrics), figsize=(6.2 * len(metrics), 5.4))
    if len(metrics) == 1:
        axes = [axes]

    labels = df["label"].tolist()
    x = list(range(len(labels)))
    for axis, (metric_key, metric_label) in zip(axes, metrics):
        values = df[metric_key].astype(float).tolist()
        axis.bar(x, values, color=["#1f4e79", "#557c55", "#4c956c", "#2f855a"][: len(labels)], width=0.72)
        axis.set_xticks(x)
        if cjk_font is not None:
            axis.set_xticklabels(labels, rotation=16, ha="right", fontproperties=cjk_font)
            axis.set_title(f"{dataset_label}: {metric_label}", fontproperties=cjk_font)
        else:
            axis.set_xticklabels(labels, rotation=16, ha="right")
            axis.set_title(f"{dataset_label}: {metric_label}")
        upper = 1.05 if not metric_key.startswith("latency") else max(values) * 1.18
        axis.set_ylim(0.0, upper)
        axis.grid(axis="y", alpha=0.2)
    figure.tight_layout()
    figure.savefig(output_path, format="svg")
    plt.close(figure)


def build_notebook(path: Path, config: dict[str, Any], rgb_df: pd.DataFrame, crud_df: pd.DataFrame) -> None:
    rgb_table = to_markdown_table(
        rgb_df,
        [
            "label",
            "candidate_top_k",
            "accuracy",
            "retrieval_hit_rate_at_1",
            "retrieval_hit_rate_at_3",
            "answer_relevancy",
            "delta_accuracy_vs_prev",
            "delta_retrieval_hit_rate_at_1_vs_prev",
            "delta_retrieval_hit_rate_at_3_vs_prev",
            "delta_answer_relevancy_vs_prev",
        ],
    )
    crud_table = to_markdown_table(
        crud_df,
        [
            "label",
            "candidate_top_k",
            "qa_accuracy",
            "qa_faithfulness",
            "qa_answer_correctness",
            "multidoc_faithfulness",
            "multidoc_answer_correctness",
            "negative_rejection",
            "delta_qa_accuracy_vs_prev",
            "delta_qa_faithfulness_vs_prev",
            "delta_multidoc_answer_correctness_vs_prev",
        ],
    )
    notebook = nbf.v4.new_notebook()
    notebook.cells = [
        nbf.v4.new_markdown_cell(
            "# RGB + CRUD 加法消融实验\n\n"
            "统一基线为 `Dense + BM25 -> RRF -> Rerank -> Direct Answer`，"
            "并与论文主线方案做正式对比。"
        ),
        nbf.v4.new_code_cell(
            "config = " + json.dumps(config, ensure_ascii=False, indent=2),
            outputs=[nbf.v4.new_output("execute_result", data={"text/plain": json.dumps(config, ensure_ascii=False, indent=2)}, execution_count=1)],
            execution_count=1,
        ),
        nbf.v4.new_markdown_cell("## RGB 结果\n\n" + rgb_table + "\n\n![RGB 消融图](../outputs/RGB300_柱状图.svg)"),
        nbf.v4.new_markdown_cell("## CRUD 结果\n\n" + crud_table + "\n\n![CRUD 消融图](../outputs/CRUD对照_柱状图.svg)"),
    ]
    notebook.metadata = {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.10.12"},
    }
    path.write_text(nbf.writes(notebook), encoding="utf-8")


def run_variant_on_cases(
    pipeline: RagExperimentPipeline,
    prepared: Any,
    variant: PipelineVariant,
) -> list[Any]:
    results = []
    total = len(prepared.cases)
    for index, case in enumerate(prepared.cases, start=1):
        if index == 1 or index % 25 == 0 or index == total:
            print(f"[{prepared.name}] {variant.key}: {index}/{total}", flush=True)
        results.append(pipeline.run_case(prepared, case, variant))
    return results


def main() -> None:
    parser = argparse.ArgumentParser(description="在 RGB 和 CRUD 上运行统一 baseline + 加法消融实验。")
    parser.add_argument("--dataset", choices=("both", "crud", "rgb"), default="both")
    parser.add_argument("--embedding-model", default=DEFAULT_EMBEDDING_MODEL)
    parser.add_argument("--llm-model", default=DEFAULT_LLM_MODEL)
    parser.add_argument("--skip-ragas", action="store_true")
    parser.add_argument("--output-root", default="", help="可选输出目录；为空时写入默认 outputs/。")
    parser.add_argument("--rgb-case-limit", type=int, default=0, help="RGB 快速验证样本数；0 为全量 300。")
    parser.add_argument("--rgb-ragas-sample-count", type=int, default=50)
    parser.add_argument("--crud-summary-samples", type=int, default=10)
    parser.add_argument("--crud-qa-1doc-samples", type=int, default=10)
    parser.add_argument("--crud-qa-2doc-samples", type=int, default=10)
    parser.add_argument("--crud-qa-3doc-samples", type=int, default=10)
    parser.add_argument("--crud-hallu-samples", type=int, default=0)
    parser.add_argument("--crud-negative-samples", type=int, default=16)
    parser.add_argument("--crud-distractor-count", type=int, default=1200)
    parser.add_argument("--crud-ragas-sample-count", type=int, default=24)
    parser.add_argument("--crud-seed", type=int, default=42)
    args = parser.parse_args()

    output_root = Path(args.output_root).resolve() if args.output_root else OUTPUT_ROOT
    notebook_root = output_root.parent / f"{output_root.name}_notebooks" if args.output_root else NOTEBOOK_ROOT

    ensure_dir(output_root)
    ensure_dir(notebook_root)
    cache_root = ensure_dir(ROOT / ".cache")
    run_rgb = args.dataset in {"both", "rgb"}
    run_crud = args.dataset in {"both", "crud"}

    pipeline = RagExperimentPipeline(
        cache_root=cache_root,
        embedding_model=args.embedding_model,
        llm_model=args.llm_model,
    )
    include_contextual = any(variant.retrieval_strategy == "contextual" for variant in VARIANTS)
    include_parent_child = any(variant.retrieval_strategy == "parent_child" for variant in VARIANTS)
    include_query_rewrite = any(variant.use_query_rewrite for variant in VARIANTS)

    rgb_cases = []
    rgb_prepared = None
    rgb_ragas_case_ids: list[str] = []
    if run_rgb:
        rgb_cases, rgb_docs = load_rgb_cases(case_limit=args.rgb_case_limit)
        rgb_prepared = pipeline.prepare_dataset(
            "rgb",
            rgb_cases,
            rgb_docs,
            include_contextual=include_contextual,
            include_parent_child=include_parent_child,
            include_query_rewrite=include_query_rewrite,
        )
        rgb_ragas_case_ids = evenly_spaced_case_ids([case.case_id for case in rgb_cases], args.rgb_ragas_sample_count)

    crud_cases = []
    crud_prepared = None
    crud_ragas_case_ids: list[str] = []
    crud_qa_ragas_case_ids: list[str] = []
    crud_multidoc_ragas_case_ids: list[str] = []
    if run_crud:
        crud_cases, crud_docs = load_crud_cases(
            summary_samples=args.crud_summary_samples,
            qa_1doc_samples=args.crud_qa_1doc_samples,
            qa_2doc_samples=args.crud_qa_2doc_samples,
            qa_3doc_samples=args.crud_qa_3doc_samples,
            hallu_samples=args.crud_hallu_samples,
            negative_samples=args.crud_negative_samples,
            distractor_count=args.crud_distractor_count,
            seed=args.crud_seed,
        )
        crud_prepared = pipeline.prepare_dataset(
            "crud",
            crud_cases,
            crud_docs,
            include_contextual=include_contextual,
            include_parent_child=include_parent_child,
            include_query_rewrite=include_query_rewrite,
        )
        crud_ragas_case_ids = evenly_spaced_case_ids(
            [case.case_id for case in crud_cases if not case.should_refuse],
            args.crud_ragas_sample_count,
        )
        crud_qa_ragas_case_ids = evenly_spaced_case_ids(
            [case.case_id for case in crud_cases if case.split.startswith("questanswer_")],
            args.crud_ragas_sample_count,
        )
        crud_multidoc_ragas_case_ids = evenly_spaced_case_ids(
            [case.case_id for case in crud_cases if case.split in {"questanswer_2docs", "questanswer_3docs"}],
            args.crud_ragas_sample_count,
        )

    rgb_summaries: list[dict[str, Any]] = []
    crud_summaries: list[dict[str, Any]] = []
    detail_rows: list[dict[str, Any]] = []
    ragas_rows: list[dict[str, Any]] = []

    for variant in VARIANTS:
        if run_rgb and rgb_prepared is not None:
            rgb_results = run_variant_on_cases(pipeline, rgb_prepared, variant)
            rgb_eval = evaluate_rgb_results(
                variant.key,
                rgb_results,
                rgb_cases,
                ragas_case_ids=rgb_ragas_case_ids,
                enable_ragas=not args.skip_ragas,
            )
            rgb_summary = dict(rgb_eval.summary)
            rgb_summary["label"] = variant.label
            rgb_summary["candidate_top_k"] = variant.candidate_top_k
            rgb_summaries.append(rgb_summary)
            detail_rows.extend(rgb_eval.detail_rows)
            ragas_rows.extend([{**row, "variant": variant.key, "dataset": "rgb"} for row in rgb_eval.ragas_rows])

        if run_crud and crud_prepared is not None:
            crud_results = run_variant_on_cases(pipeline, crud_prepared, variant)
            crud_eval = evaluate_crud_results(
                variant.key,
                crud_results,
                crud_cases,
                ragas_case_ids=crud_ragas_case_ids,
                qa_ragas_case_ids=crud_qa_ragas_case_ids,
                multidoc_ragas_case_ids=crud_multidoc_ragas_case_ids,
                enable_ragas=not args.skip_ragas,
                semantic_model_name=args.embedding_model,
            )
            crud_summary = dict(crud_eval.summary)
            crud_summary["label"] = variant.label
            crud_summary["candidate_top_k"] = variant.candidate_top_k
            crud_summaries.append(crud_summary)
            detail_rows.extend(crud_eval.detail_rows)
            ragas_rows.extend([{**row, "variant": variant.key, "dataset": "crud"} for row in crud_eval.ragas_rows])

    summary_payload: dict[str, Any] = {}
    rgb_df = pd.DataFrame()
    crud_df = pd.DataFrame()
    if run_rgb and rgb_summaries:
        rgb_df = add_deltas(
            pd.DataFrame(rgb_summaries),
            ["accuracy", "retrieval_hit_rate_at_1", "retrieval_hit_rate_at_3", "answer_relevancy"],
        )
        rgb_df.to_csv(output_root / "RGB300_指标表.csv", index=False, encoding="utf-8")
        summary_payload["rgb"] = rgb_df.to_dict(orient="records")
    if run_crud and crud_summaries:
        crud_df = add_deltas(
            pd.DataFrame(crud_summaries),
            [
                "qa_accuracy",
                "qa_faithfulness",
                "qa_answer_correctness",
                "multidoc_faithfulness",
                "multidoc_answer_correctness",
                "negative_rejection",
                "overall_similarity",
                "integration_similarity",
            ],
        )
        crud_df.to_csv(output_root / "CRUD对照_指标表.csv", index=False, encoding="utf-8")
        summary_payload["crud"] = crud_df.to_dict(orient="records")

    pd.DataFrame(detail_rows).to_csv(output_root / "支撑实验_逐题明细.csv", index=False, encoding="utf-8")
    pd.DataFrame(ragas_rows).to_csv(output_root / "支撑实验_RAGAS明细.csv", index=False, encoding="utf-8")

    config = {
        "dataset": args.dataset,
        "baseline": VARIANTS[0].label,
        "variants": [variant.__dict__ for variant in VARIANTS],
        "embedding_model": args.embedding_model,
        "llm_model": args.llm_model,
        "rgb_case_limit": args.rgb_case_limit,
        "rgb_ragas_sample_count": args.rgb_ragas_sample_count,
        "crud_summary_samples": args.crud_summary_samples,
        "crud_qa_1doc_samples": args.crud_qa_1doc_samples,
        "crud_qa_2doc_samples": args.crud_qa_2doc_samples,
        "crud_qa_3doc_samples": args.crud_qa_3doc_samples,
        "crud_hallu_samples": args.crud_hallu_samples,
        "crud_negative_samples": args.crud_negative_samples,
        "crud_distractor_count": args.crud_distractor_count,
        "crud_ragas_sample_count": args.crud_ragas_sample_count,
        "skip_ragas": args.skip_ragas,
    }
    (output_root / "支撑实验_实验配置.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_root / "支撑实验_汇总.json").write_text(
        json.dumps(summary_payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    if run_rgb and not rgb_df.empty:
        plot_dataset_metrics(
            rgb_df,
            output_root / "RGB300_柱状图.svg",
            dataset_label="RGB",
            metrics=(
                ("accuracy", "准确率"),
                ("retrieval_hit_rate_at_1", "命中率 Hit@1"),
                ("retrieval_hit_rate_at_3", "命中率 Hit@3"),
            ),
        )
    if run_crud and not crud_df.empty:
        plot_dataset_metrics(
            crud_df,
            output_root / "CRUD对照_柱状图.svg",
            dataset_label="CRUD",
            metrics=(
                ("qa_accuracy", "问答准确率"),
                ("qa_faithfulness", "答案忠实度"),
                ("multidoc_answer_correctness", "多文档答案正确率"),
            ),
        )

    if run_rgb and run_crud and not rgb_df.empty and not crud_df.empty:
        build_notebook(notebook_root / "rgb_crud_ablation.ipynb", config, rgb_df, crud_df)
    print(json.dumps(summary_payload, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 结果预览

下面直接内嵌当前已保存结果的关键文件预览。

### 实验配置

- 文件：`../results/03_RGB300支撑结果/支撑实验_实验配置.json`

In [1]:
from pathlib import Path
import json

path = Path('../results/03_RGB300支撑结果/支撑实验_实验配置.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "baseline": "基线（Dense + BM25 -> RRF -> Rerank -> Direct Answer）",
  "variants": [
    {
      "key": "baseline_rrf_rerank_direct",
      "label": "基线（Dense + BM25 -> RRF -> Rerank -> Direct Answer）",
      "retrieval_strategy": "raw",
      "use_rerank": true,
      "use_query_rewrite": false,
      "rerank_mode": "default",
      "synthesis_mode": "direct",
      "answer_prompt_style": "simple",
      "evidence_selection": "topk",
      "adaptive_prompt_min_query_tokens": 0,
      "multi_snippet_count": 1,
      "retrieval_top_k": 50,
      "candidate_top_k": 20,
      "final_source_count": 3,
      "complex_source_count": 5,
      "distilled_fact_limit": 6,
      "router_complexity_threshold": 0.5,
      "context_window_sentences": 0,
      "compression_max_units": 6,
      "support_pruning_threshold": 0.0,
      "span_constraint_limit": 6,
      "selection_mode": "default",
      "rendering_mode": "default"
    },
    {
      "key": "enhanced_dual_snippet_direct",
      "label":

### 实验汇总

- 文件：`../results/03_RGB300支撑结果/支撑实验_汇总.json`

In [2]:
from pathlib import Path
import json

path = Path('../results/03_RGB300支撑结果/支撑实验_汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "rgb": [
    {
      "variant": "baseline_rrf_rerank_direct",
      "dataset": "rgb",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.9733,
      "retrieval_hit_rate_at_1": 0.9233,
      "retrieval_hit_rate_at_3": 0.9867,
      "latency_p50_ms": 817.41,
      "latency_p95_ms": 1554.32,
      "sample_count": 300,
      "label": "基线（Dense + BM25 -> RRF -> Rerank -> Direct Answer）",
      "candidate_top_k": 20,
      "delta_accuracy_vs_prev": "baseline",
      "delta_retrieval_hit_rate_at_1_vs_prev": "baseline",
      "delta_retrieval_hit_rate_at_3_vs_prev": "baseline",
      "delta_answer_relevancy_vs_prev": "baseline"
    },
    {
      "variant": "enhanced_dual_snippet_direct",
      "dataset": "rgb",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
     

### RGB300 指标表

- 文件：`../results/03_RGB300支撑结果/RGB300_指标表.csv`

In [3]:
from pathlib import Path
import csv

path = Path('../results/03_RGB300支撑结果/RGB300_指标表.csv')
with path.open('r', encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
print(f'rows={len(rows)} preview={min(len(rows), 8)}')
for row in rows[:8]:
    print(row)


variant,dataset,faithfulness,answer_correctness,answer_relevancy,context_precision,ragas_sample_count,accuracy,retrieval_hit_rate_at_1,retrieval_hit_rate_at_3,latency_p50_ms,latency_p95_ms,sample_count,label,candidate_top_k,delta_accuracy_vs_prev,delta_retrieval_hit_rate_at_1_vs_prev,delta_retrieval_hit_rate_at_3_vs_prev,delta_answer_relevancy_vs_prev
baseline_rrf_rerank_direct,rgb,0.0,0.0,0.0,0.0,0,0.9733,0.9233,0.9867,817.41,1554.32,300,基线（Dense + BM25 -> RRF -> Rerank -> Direct Answer）,20,baseline,baseline,baseline,baseline
enhanced_dual_snippet_direct,rgb,0.0,0.0,0.0,0.0,0,0.97,0.9233,0.9867,221.34,322.25,300,增强检索（双片段证据保留）,20,-0.0033,+0.0000,+0.0000,+0.0000
ours_task_aligned_ms2,rgb,0.0,0.0,0.0,0.0,0,0.9633,0.9233,0.9867,770.12,1696.66,300,Ours（+ Dual-Snippet Rerank + Aspect-Aware Rerank V2 + Task-对齐作答 Answer）,20,-0.0067,+0.0000,+0.0000,+0.0000


### CRUD 对照指标表

- 文件：`../results/03_RGB300支撑结果/CRUD对照_指标表.csv`

In [4]:
from pathlib import Path
import csv

path = Path('../results/03_RGB300支撑结果/CRUD对照_指标表.csv')
with path.open('r', encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
print(f'rows={len(rows)} preview={min(len(rows), 8)}')
for row in rows[:8]:
    print(row)


variant,dataset,faithfulness,answer_correctness,answer_relevancy,context_precision,ragas_sample_count,accuracy,qa_accuracy,retrieval_hit_rate_at_1,retrieval_hit_rate_at_3,qa_faithfulness,qa_answer_correctness,qa_answer_relevancy,qa_context_precision,qa_ragas_sample_count,multidoc_faithfulness,multidoc_answer_correctness,multidoc_answer_relevancy,multidoc_context_precision,multidoc_ragas_sample_count,overall_similarity,summary_similarity,noise_robustness,negative_rejection,information_integration,integration_similarity,latency_p50_ms,latency_p95_ms,sample_count,label,candidate_top_k,delta_qa_accuracy_vs_prev,delta_qa_faithfulness_vs_prev,delta_qa_answer_correctness_vs_prev,delta_multidoc_faithfulness_vs_prev,delta_multidoc_answer_correctness_vs_prev,delta_negative_rejection_vs_prev,delta_overall_similarity_vs_prev,delta_integration_similarity_vs_prev
baseline_rrf_rerank_direct,crud,0.0,0.0,0.0,0.0,0,0.02,0.0333,0.72,0.88,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.4588,0.4574,0.1709,1.0,0.0,0.4662,1963.1,3524.03,66,基线（Dense + BM25 -> RRF -> Rerank -> Direct Answer）,20,baseline,baseline,baseline,baseline,baseline,baseline,baseline,baseline
enhanced_dual_snippet_direct,crud,0.0,0.0,0.0,0.0,0,0.02,0.0333,0.74,0.84,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.4233,0.4578,0.0863,1.0,0.0,0.4184,2394.83,4214.56,66,增强检索（双片段证据保留）,20,+0.0000,+0.0000,+0.0000,+0.0000,+0.0000,+0.0000,-0.0355,-0.0478
ours_task_aligned_ms2,crud,0.0,0.0,0.0,0.0,0,0.04,0.0667,0.76,0.82,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.5036,0.4192,0.2883,1.0,0.0,0.5322,2302.79,3669.89,66,Ours（+ Dual-Snippet Rerank + Aspect-Aware Rerank V2 + Task-对齐作答 Answer）,20,+0.0334,+0.0000,+0.0000,+0.0000,+0.0000,+0.0000,+0.0803,+0.1138


### RGB300 柱状图

- 文件：`../results/03_RGB300支撑结果/RGB300_柱状图.svg`

**图像文件**：`../results/03_RGB300支撑结果/RGB300_柱状图.svg`

![RGB300 柱状图](../results/03_RGB300支撑结果/RGB300_柱状图.svg)

### CRUD 对照柱状图

- 文件：`../results/03_RGB300支撑结果/CRUD对照_柱状图.svg`

**图像文件**：`../results/03_RGB300支撑结果/CRUD对照_柱状图.svg`

![CRUD 对照柱状图](../results/03_RGB300支撑结果/CRUD对照_柱状图.svg)